# Tool Calling

- Built in Web-search [openai web-search](https://platform.openai.com/docs/guides/tools-web-search)
- Get Weather - mock function call 
- A2AJ Canadian Legal MCP Server [A2AJ MCP](https://github.com/a2aj-ca/canadian-legal-data/blob/main/access-via-mcp.ipynb)

In [2]:
from dotenv import load_dotenv
import os
from openai import OpenAI, AsyncOpenAI

load_dotenv(override=True)

if (OPENAI_API_KEY := os.getenv("OPENAI_API_KEY")) is None:
    raise ValueError("OPENAI_API_KEY not found in environment variables")


In [28]:
tools = [
    { "type": "web_search" },
    { 
        "type": "function", 
        "name": "get_weather",
        "description": "Get the current weather for a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country, e.g. 'London, UK'"
                }
            },
            "required": ["location"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "mcp",
        "server_label": "a2aj",
        "server_url": "https://api.a2aj.ca/mcp",
        "require_approval": "never",
    }

]

In [3]:
import json

async def get_weather(location: str) -> str:
    """Fetches the current weather for a given location."""

    return {
        "location": location,
        "temperature": "22°C",
        "condition": "Partly Cloudy"
    }

async def execute_function_tool(name: str, arguments: str):
    args = json.loads(arguments)
    if name == "get_weather":
        location = args.get("location")
        result = await get_weather(location)
        return result
    
    

In [3]:
system_prompt = """You are a helpful, reliable AI assistant with access to external tools. Your goal is to answer user queries accurately by reasoning first and using tools only when necessary.

## Tool Usage Rules

### web_search
- Use only when the user asks for recent, real-time, or time-sensitive information.
- Use when the information is likely not included in your training data.
- Prefer internal knowledge when sufficient.
- Summarize findings clearly and avoid fabricating facts.

### get_weather
- Use only when the user asks about the current weather or conditions for a specific location.
- If the location is missing or ambiguous, ask a clarifying question before calling the tool.
- Do not guess or infer weather information without using the tool.

### a2aj MCP
- Use only for queries involving legal cases, legal processes, rights, obligations, or other legal scenarios.
- Delegate legal reasoning and explanations to the a2aj MCP instead of answering directly.
- Do not provide legal advice without using the a2aj MCP.

## Reasoning and Behavior
- Always decide whether a tool is required before responding.
- Never fabricate information that should come from a tool.
- If multiple tools could apply, select the most authoritative one.
- Keep responses concise, neutral, and user-friendly.

## Response Guidelines
- If no tool is needed, respond directly.
- If a tool is needed, call it before responding.
- After using a tool, explain the result clearly in plain language.

"""

## Responses API

In [29]:
from uuid import uuid4
from openai import AsyncOpenAI
from openai.types.responses import ResponseInputParam, ResponseTextDeltaEvent, ResponseOutputItemDoneEvent

conversation = [
    { 
        "type": "message",
        "role": "user", 
        "content": [
            {
                "type": "input_text",
                "text": """Briefly summarize the latest Canadian Human Rights Tribunal decision involving racial discrimination.
Notes: 
- Read the full decision before answering the question.
- You can make multiple calls to get the full text."""
            }
        ],
        "status": "completed"
    }
]


try:
    llm_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    
    turn = 0
    max_turns = 5

    while turn < max_turns:
        print(f"\n\\--- Turn {turn+1} ---\\\n")

        response_stream = await llm_client.responses.create(
            model="gpt-5-mini",
            input=conversation,
            instructions=system_prompt,
            tools=tools,
            stream=True,
            store=False
        )

        assistant_content = ""
        function_call_made = False

        async for event in response_stream:

            if isinstance(event, ResponseTextDeltaEvent):
                assistant_content += event.delta
                print(event.delta, end="", flush=True)
            elif isinstance(event, ResponseOutputItemDoneEvent):
                item = event.item

                if item.type == "message":
                    conversation.append(item.model_dump())
                elif item.type == "web_search_call":
                    conversation.append(item.model_dump())    #TODO: check if this is needed?
                    print(f"\n --- Web Search Call - action: {item.action.model_dump()} - status: {item.status} ---\n")
                elif item.type == "function_call":
                    conversation.append(item.model_dump())
                    print(f"\n --- Function Call - name: {item.name} - args: {item.arguments} - status: {item.status} ---\n")
                    function_output = await execute_function_tool(name=item.name, arguments=item.arguments)
                    conversation.append({
                        "call_id": item.call_id,
                        "output": str(function_output),
                        "type": "function_call_output",
                        # "id": uuid4(),  
                        # "status": "completed"
                    })
                    function_call_made = True
                elif item.type == "mcp_list_tools":
                    conversation.append(item.model_dump())
                    print(f"\n --- MCP List Tools - server: {item.server_label} - tools: {[tool.name for tool in item.tools]} ---\n")
                elif item.type == "mcp_call":
                    conversation.append(item.model_dump())
                    print(f"\n --- MCP Call - server: {item.server_label} - tool: {item.name} ---\n")
            
        if not function_call_made:
            break
            
        turn += 1

except Exception as e:
    print(f"[ERROR] - {str(e)}")
finally:
    await llm_client.close()


\--- Turn 1 ---\


 --- MCP List Tools - server: a2aj - tools: ['coverage', 'fetch', 'search'] ---


 --- MCP Call - server: a2aj - tool: search ---


 --- MCP Call - server: a2aj - tool: search ---


 --- MCP Call - server: a2aj - tool: fetch ---


 --- MCP Call - server: a2aj - tool: search ---


 --- MCP Call - server: a2aj - tool: fetch ---


 --- MCP Call - server: a2aj - tool: fetch ---


 --- MCP Call - server: a2aj - tool: search ---


 --- MCP Call - server: a2aj - tool: fetch ---

Short answer — and I read the full text before replying.

- Case: Richards v. Correctional Service Canada, 2025 CHRT 107 (Tribunal Member Jennifer Khurana). Decision dated November 12, 2025.  
- What the complaint alleged: the complainant (Ryan Richards) raised claims including colour/race (and religion, sex) and retaliation against Correctional Service Canada.  
- Tribunal outcome: the Tribunal dismissed the complaints in their entirety — not on the merits of the discrimination allegations, but as

In [30]:
print(json.dumps(conversation, indent=2))
await llm_client.close()

[
  {
    "type": "message",
    "role": "user",
    "content": [
      {
        "type": "input_text",
        "text": "Briefly summarize the latest Canadian Human Rights Tribunal decision involving racial discrimination.\nNotes: \n- Read the full decision before answering the question.\n- You can make multiple calls to get the full text."
      }
    ],
    "status": "completed"
  },
  {
    "id": "mcpl_01d90718f2a3016601697cd1dd84448190be42f1d228eb0fa7",
    "server_label": "a2aj",
    "tools": [
      {
        "input_schema": {
          "type": "object",
          "properties": {
            "doc_type": {
              "enum": [
                "cases",
                "laws"
              ],
              "type": "string",
              "description": "'cases' (default) for case law or 'laws' for statutes & regulations",
              "default": "cases",
              "title": "doc_type"
            }
          },
          "title": "coverageArguments"
        },
        "name":

## Agents SDK

In [1]:
from agents import function_tool

@function_tool
async def get_weather(location: str) -> str:
    """Fetches the current weather for a given location."""

    return str({
        "location": location,
        "temperature": "22°C",
        "condition": "Partly Cloudy"
    })

In [ ]:
from agents.run import RunConfig
from agents import (
    Agent, 
    OpenAIResponsesModel, 
    Runner, 
    WebSearchTool, 
    HostedMCPTool, 
    RawResponsesStreamEvent, 
    RunItemStreamEvent, 
    AgentUpdatedStreamEvent
)

try:
    openai_gpt5_mini = OpenAIResponsesModel(
        model="gpt-5-mini",
        openai_client=AsyncOpenAI(api_key=OPENAI_API_KEY)
    )

    assistant_agent = Agent(
        name="Assistant",
        instructions=system_prompt,
        model=openai_gpt5_mini,
        tools=[
            WebSearchTool(),
            get_weather,
            HostedMCPTool(
                tool_config={
                    "type": "mcp",
                    "server_label": "a2aj",
                    "server_url": "https://api.a2aj.ca/mcp",
                    "require_approval": "never",
                }
            )
        ]
    )

    conversation = [
        { 
            "type": "message",
            "role": "user", 
            "content": [
                {
                    "type": "input_text",
                    "text": "Give me the current weather in Toronto, Canada."
                }
            ],
            "status": "completed"
        }
    ]

    agent_response_stream = Runner.run_streamed(
        starting_agent=assistant_agent,
        input=conversation,
        run_config=RunConfig(
            tracing_disabled=True
        )
    )

    async for event in agent_response_stream.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if data.type == "response.output_text.delta":
                print(event.data.delta, end="", flush=True)
            elif data.type == "response.output_text.done":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")
            elif data.type == "response.web_search_call.searching":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")
            elif data.type == "response.web_search_call.completed":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")
            elif data.type == "response.function_call_arguments.done":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} - arguments: {data.arguments} --")
            elif data.type == "response.mcp_list_tools.in_progress":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")
            elif data.type == "response.mcp_list_tools.completed":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")
            elif data.type == "response.mcp_call.in_progress":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")
            elif data.type == "response.mcp_call.completed":
                print(f" -- [RAW RESPONSE STREAM EVENT][{data.output_index}] - {data.type} --")

        elif isinstance(event, RunItemStreamEvent):
            print(f"--- [RUN ITEM STREAM EVENT] {event.name} - item.type: {event.item.type} ---")
            if event.item.type == "tool_call_item":
                item = event.item.raw_item
                print(f"    -- {item.type} - status: {item.status} --")

        elif isinstance(event, AgentUpdatedStreamEvent):
            print(f"--- [AGENT UPDATED STREAM EVENT] {event.type} - {event.new_agent.name} ---")


except Exception as e:
    print(f"[ERROR] - {str(e)}")

--- [AGENT UPDATED STREAM EVENT] agent_updated_stream_event - Assistant ---
 -- [RAW RESPONSE STREAM EVENT][0] - response.mcp_list_tools.in_progress --
 -- [RAW RESPONSE STREAM EVENT][0] - response.mcp_list_tools.completed --
--- [RUN ITEM STREAM EVENT] reasoning_item_created - item.type: reasoning_item ---
 -- [RAW RESPONSE STREAM EVENT][2] - response.function_call_arguments.done - arguments: {"location":"Toronto, Canada"} --
--- [RUN ITEM STREAM EVENT] tool_called - item.type: tool_call_item ---
    -- function_call - status: completed --
--- [RUN ITEM STREAM EVENT] mcp_list_tools - item.type: mcp_list_tools_item ---
--- [RUN ITEM STREAM EVENT] tool_output - item.type: tool_call_output_item ---
Current weather in Toronto, Canada:
- Temperature: 22°C
- Condition: Partly cloudy

Would you like a short forecast (next few hours), precipitation chance, or temperature in °F? -- [RAW RESPONSE STREAM EVENT][0] - response.output_text.done --
--- [RUN ITEM STREAM EVENT] message_output_created 

In [12]:
import json
print(json.dumps(conversation, indent=2))

[
  {
    "type": "message",
    "role": "user",
    "content": [
      {
        "type": "input_text",
        "text": "Give me the current weather in Toronto, Canada."
      }
    ],
    "status": "completed"
  }
]
